<a href="https://colab.research.google.com/github/vivekworldwide/Deep-learning-Notes/blob/main/pytorch_rnn_based_qa_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd

df = pd.read_csv('/content/100_Unique_QA_Dataset.csv')

df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [3]:
# tokenize
def tokenize(text):
  text=text.lower()
  text = text.replace('?', '')
  text = text.replace("'", "")
  return text.split()


In [4]:
tokenize("'Who wrote 'To Kill a Mockingbird'?'")

['who', 'wrote', 'to', 'kill', 'a', 'mockingbird']

In [5]:
# vocab
vocab = {'<UNK>':0}

In [6]:
def build_vocab(row):
  tokenized_question= tokenize(row['question'])
  tokenized_answer = tokenize(row['answer'])
  merged_tokens = tokenized_question+tokenized_answer
  print(merged_tokens)

  for token in merged_tokens:

    if token not in vocab:
      vocab[token]= len(vocab)

In [7]:
df.apply(build_vocab, axis= 1)

['what', 'is', 'the', 'capital', 'of', 'france', 'paris']
['what', 'is', 'the', 'capital', 'of', 'germany', 'berlin']
['who', 'wrote', 'to', 'kill', 'a', 'mockingbird', 'harper-lee']
['what', 'is', 'the', 'largest', 'planet', 'in', 'our', 'solar', 'system', 'jupiter']
['what', 'is', 'the', 'boiling', 'point', 'of', 'water', 'in', 'celsius', '100']
['who', 'painted', 'the', 'mona', 'lisa', 'leonardo-da-vinci']
['what', 'is', 'the', 'square', 'root', 'of', '64', '8']
['what', 'is', 'the', 'chemical', 'symbol', 'for', 'gold', 'au']
['which', 'year', 'did', 'world', 'war', 'ii', 'end', '1945']
['what', 'is', 'the', 'longest', 'river', 'in', 'the', 'world', 'nile']
['what', 'is', 'the', 'capital', 'of', 'japan', 'tokyo']
['who', 'developed', 'the', 'theory', 'of', 'relativity', 'albert-einstein']
['what', 'is', 'the', 'freezing', 'point', 'of', 'water', 'in', 'fahrenheit', '32']
['which', 'planet', 'is', 'known', 'as', 'the', 'red', 'planet', 'mars']
['who', 'is', 'the', 'author', 'of', '19

,0
0,None
1,None
2,None
3,None
4,None
...,...
85,None
86,None
87,None
88,None


In [8]:
len(vocab)


324

In [9]:
from torch import index_reduce
# convert words to numerical indices
def text_to_indices(text, vocab):

  indexed_text =[]

  for token in tokenize(text):
    if token in vocab:
      indexed_text.append(vocab[token])
    else:
      indexed_text.append(vocab['<UNK>'])
  return indexed_text

In [10]:
text_to_indices("What is low value", vocab)

[1, 2, 0, 0]

In [11]:
import torch
from torch.utils.data import Dataset, DataLoader

In [12]:
class QADataset(Dataset):

  def __init__(self, df, vocab):
    self.df = df
    self.vocab = vocab

  def __len__(self):
    return self.df.shape[0]

  def __getitem__(self, index):
    numerical_question = text_to_indices(self.df.iloc[index]['question'], self.vocab)
    numerical_answer = text_to_indices(self.df.iloc[index]['answer'], self.vocab)

    return torch.tensor(numerical_question), torch.tensor(numerical_answer)

In [13]:
dataset = QADataset(df, vocab)

In [14]:
dataset[1]

(tensor([1, 2, 3, 4, 5, 8]), tensor([9]))

In [15]:

dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

for question, answer in dataloader:
  print(question, answer)

tensor([[  1,   2,   3,  37,  38,  39, 161]]) tensor([[162]])
tensor([[ 10,  11, 157, 158, 159]]) tensor([[160]])
tensor([[ 78,  79, 261, 151,  14, 262, 153]]) tensor([[36]])
tensor([[  1,   2,   3, 141, 117,  83,   3, 277, 278]]) tensor([[121]])
tensor([[  1,   2,   3,   4,   5, 135]]) tensor([[136]])
tensor([[  1,   2,   3,  69,   5, 155]]) tensor([[156]])
tensor([[  1,   2,   3,  37, 133,   5,  26]]) tensor([[134]])
tensor([[10, 96,  3, 97]]) tensor([[98]])
tensor([[ 10, 308,   3, 309, 310]]) tensor([[311]])
tensor([[42, 43, 44, 45, 46, 47, 48]]) tensor([[49]])
tensor([[10, 11, 12, 13, 14, 15]]) tensor([[16]])
tensor([[  1,   2,   3, 234,   5, 235]]) tensor([[131]])
tensor([[ 10,  75, 208]]) tensor([[209]])
tensor([[ 1,  2,  3, 37, 38, 39, 40]]) tensor([[41]])
tensor([[  1,   2,   3, 146,  86,  19, 192, 193]]) tensor([[194]])
tensor([[ 1,  2,  3, 69,  5,  3, 70, 71]]) tensor([[72]])
tensor([[ 42, 137,   2, 138,  39, 175, 269]]) tensor([[99]])
tensor([[ 10,  96,   3, 104, 239]]) tens

In [16]:
import torch.nn as nn

In [17]:
class SimpleRNN(nn.Module):

  def __init__ (self, vocab_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim=50)
    self.rnn = nn.RNN(50,64, batch_first = True)
    self.fc = nn.Linear(64,vocab_size)

  def forward(self, question):
    embedded_question = self.embedding(question)
    hidden, final = self.rnn(embedded_question)
    output = self.fc(final.squeeze(0))

    return output



In [18]:
learning_rate = 0.001
epochs = 20


In [19]:
model = SimpleRNN(len(vocab))


In [20]:
criterian = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = learning_rate)

In [22]:
import torch.nn.functional as F
#training loop
for epoch in range(epochs):

  total_loss= 0
  for question, answer in dataloader:

    optimizer.zero_grad()

    #forward  pass
    output = model(question)

    #loss
    loss = criterian(output, answer.squeeze(-1))

    #gradient
    loss.backward()

    #weight update
    optimizer.step()

    total_loss = total_loss + loss.item()

  print(f"Epoch: {epoch+1}, loss: {total_loss:.4f}")

Epoch: 1, loss: 523.5792
Epoch: 2, loss: 454.3348
Epoch: 3, loss: 372.5498
Epoch: 4, loss: 312.2115
Epoch: 5, loss: 261.2336
Epoch: 6, loss: 213.7459
Epoch: 7, loss: 169.6023
Epoch: 8, loss: 130.9747
Epoch: 9, loss: 100.3221
Epoch: 10, loss: 76.6127
Epoch: 11, loss: 58.6136
Epoch: 12, loss: 46.0733
Epoch: 13, loss: 36.2741
Epoch: 14, loss: 29.5734
Epoch: 15, loss: 24.3803
Epoch: 16, loss: 20.3005
Epoch: 17, loss: 17.0465
Epoch: 18, loss: 14.3969
Epoch: 19, loss: 12.3407
Epoch: 20, loss: 10.6409


In [ ]:
def predict(model, question, threshold=0.5):

  # convert question to numbers
  numerical_question = text_to_indices(question, vocab)

  # tensor
  question_tensor = torch.tensor(numerical_question).unsqueeze(0)

  # send to model
  output = model(question_tensor)

  # convert logits to probs
  probs = torch.nn.functional.softmax(output, dim=1)

  # find index of max prob
  value, index = torch.max(probs, dim=1)

  if value < threshold:
    print("I don't know")

  print(list(vocab.keys())[index])

In [ ]:
predict(model, "What is the largest planet in our solar system?")

In [ ]:
list(vocab.keys())[7]